# Fine-tune Whisper `small` cho tiếng Nhật (Colab T4) — tuỳ chọn

Notebook này fine-tune `openai/whisper-small` cho nhận dạng giọng nói tiếng Nhật (ASR), so sánh **CER** (tỉ lệ lỗi ký tự)
giữa model gốc và model fine-tune trên tập test giữ riêng, rồi xuất sang **faster-whisper (CTranslate2 int8)** và nén thành
`whisper-ja-ct2-int8.zip` để dùng trong app.

**Thứ tự:** cài thư viện → cấu hình → đăng nhập Hugging Face → mount Drive + resume → tải dữ liệu (Common Voice ja +
ReazonSpeech, tập con nhỏ) → huấn luyện `Seq2SeqTrainer` → CER gốc vs fine-tune → xuất CTranslate2 int8 → zip và tải về.

**Yêu cầu:** Runtime → Change runtime type → **T4 GPU**. Checkpoint huấn luyện ghi ra ổ local `/content` và chỉ chép lên
Google Drive (`MyDrive/whisper_ja_finetune`) một số lần có giới hạn, nên khi Colab ngắt kết nối chỉ cần **Run all** lại:
huấn luyện chạy tiếp từ checkpoint cuối trên Drive (dữ liệu âm thanh được tải và giải mã lại, khoảng 10–15 phút).

**Chạy thử nhanh (smoke test):** đặt `SMOKE_TEST = True` ở ô cấu hình (50 bước, vài trăm câu mỗi nguồn) để kiểm tra toàn bộ
quy trình ra được file zip trước khi chạy thật (`MAX_STEPS = 2000`, khoảng 1,5–2,5 giờ trên T4).

**License:** model gốc Whisper là Apache-2.0. Common Voice là CC0-1.0. ReazonSpeech là CDLA-Sharing-1.0 **và chỉ được dùng
cho mục đích theo Điều 30-4 Luật Bản quyền Nhật Bản** (phân tích thông tin/học máy); model fine-tune có dùng ReazonSpeech cũng
nên dùng theo đúng điều kiện đó. Chi tiết ở `training/data_sources.md`.

## 1. Cài thư viện
Torch, pyarrow và huggingface_hub có sẵn trên Colab nên không cài lại. `faster-whisper` (kèm PyAV để giải mã mp3/flac và
resample về 16 kHz) dùng để chạy thử model đã xuất giống hệt app. Nếu pip báo xung đột phiên bản, có thể bỏ số phiên bản
(`==...`) rồi chạy lại.

In [ ]:
!pip -q install "transformers==5.21.0" "accelerate==1.15.0" "ctranslate2==4.8.4" "faster-whisper==1.2.1"

## 2. Cấu hình
Bật/tắt từng nguồn dữ liệu bằng `True`/`False`.

| Khoá | Nguồn | License / điều kiện | Cần đăng nhập HF | Mặc định |
|---|---|---|---|---|
| `common_voice_ja` | Common Voice 17.0 tiếng Nhật (bản parquet trên Hugging Face) | CC0-1.0; không được tìm cách xác định danh tính người nói | không | bật |
| `reazonspeech` | ReazonSpeech v2 (giọng nói từ truyền hình Nhật), tập `small` | CDLA-Sharing-1.0, **chỉ dùng theo Điều 30-4 Luật Bản quyền Nhật** | **có** (phải chấp nhận điều khoản) | bật |

Mặc định: ~16.000 câu Common Voice (split `train` + `validation`) + 8.000 câu ReazonSpeech (2 file parquet đầu).
`MAX_STEPS = 2000` × batch 8 × tích luỹ gradient 2 = 32.000 mẫu ≈ 1 epoch.

In [ ]:
# CELL: config
SMOKE_TEST = False            # True: 50 steps on a few hundred clips per source (end-to-end check)
DATASETS = {
    "common_voice_ja": True,  # Common Voice 17.0 ja (CC0-1.0), no login needed
    "reazonspeech": True,     # ReazonSpeech v2 (CDLA-Sharing-1.0, Art. 30-4 only); HF login + accepted terms
}
CV_MAX_TRAIN = 16_000         # Common Voice clips for training (train + validation splits)
REAZON_TRAIN_SHARDS = 2       # ReazonSpeech "small" parquet shards read for training (~4.1k clips, ~470 MB each)
REAZON_MAX_TRAIN = 8_000
EVAL_PER_SOURCE = 300         # held-out test clips per source for CER
MIN_SEC, MAX_SEC = 0.5, 30.0  # Whisper sees at most 30 s per input
MAX_LABEL_TOKENS = 440        # decoder limit is 448 tokens including the prefix tokens

BASE_SIZE = "small"
BASE_MODEL = "openai/whisper-small"
MAX_STEPS = 2000
LR = 1e-5
BATCH = 8                     # per-device batch (T4, fp16 + gradient checkpointing)
GRAD_ACCUM = 2
FP16 = True
OPTIM = "adamw_torch"         # AdamW state => ~2.9 GB per checkpoint (242M params)
WARMUP_STEPS = 200
SAVE_STEPS = 500              # local save interval; a save may be copied to ONE fixed Drive folder (replaced)
DRIVE_SYNC_MIN = 30           # copy a checkpoint to Drive at most once per this many minutes (first save always)
DRIVE_SYNC_MAX = 2            # max Drive copies per session (each replaced copy, ~2.9 GB, sits in Drive Trash)
SEED = 42
SAMPLE_RATE = 16000
EVAL_BATCH = 16

if SMOKE_TEST:
    MAX_STEPS, SAVE_STEPS, WARMUP_STEPS = 50, 25, 5
    CV_MAX_TRAIN, REAZON_TRAIN_SHARDS, REAZON_MAX_TRAIN, EVAL_PER_SOURCE = 400, 1, 400, 50

# Pinned dataset revisions (checked 2026-10-28).
CV_REPO, CV_REV = "fixie-ai/common_voice_17_0", "34f78a43893414e7b6e271ba94c1d5e05f18b239"
REAZON_GATE = "reazon-research/reazonspeech"  # official gated page: access = terms accepted
REAZON_REPO = "japanese-asr/whisper_transcriptions.reazonspeech.small"
REAZON_REV = "c74b52fc164cf7b64936ca62aee4336eac626739"
REAZON_SHARDS = 15            # small/train-000NN-of-00015.parquet; the last shard is the test shard

DRIVE_DIR = "/content/drive/MyDrive/whisper_ja_finetune"
EXPORT_NAME = "small-ja"      # folder name under models/asr/ and the value of asr.model in the app
ZIP_NAME = "whisper-ja-ct2-int8.zip"

## 3. Đăng nhập Hugging Face (cần cho ReazonSpeech)
1. Đăng nhập https://huggingface.co, mở https://huggingface.co/datasets/reazon-research/reazonspeech, đọc điều khoản và bấm
   chấp nhận (đồng ý chỉ dùng dữ liệu theo Điều 30-4 Luật Bản quyền Nhật Bản).
2. Tạo token loại **Read** tại https://huggingface.co/settings/tokens.
3. Cách khuyên dùng: thêm token vào **Colab Secrets** (biểu tượng chìa khoá ở thanh bên trái) với tên `HF_TOKEN` và bật
   "Notebook access". Nếu không, ô dưới sẽ hỏi token (dán vào rồi Enter; để trống = bỏ qua ReazonSpeech).

Không dán token trực tiếp vào code. Common Voice không cần đăng nhập.

In [ ]:
# CELL: hf_login
import os
from getpass import getpass
from huggingface_hub import get_token, login

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:  # not on Colab, or the secret is missing / not shared with this notebook
        HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
elif DATASETS["reazonspeech"] and not get_token():
    entered = getpass("Hugging Face token (Enter để bỏ qua ReazonSpeech): ").strip()
    if entered:
        login(token=entered, add_to_git_credential=False)
print("Đã đăng nhập Hugging Face" if get_token() else "Chưa đăng nhập Hugging Face")


def reazon_access() -> bool:
    # Access to the official gated page = terms accepted. Network/5xx errors are re-raised.
    from huggingface_hub import auth_check
    from huggingface_hub.errors import GatedRepoError, RepositoryNotFoundError
    try:
        auth_check(REAZON_GATE, repo_type="dataset")
        return True
    except (GatedRepoError, RepositoryNotFoundError) as e:
        print(f"[reazonspeech] BỎ QUA: chưa có quyền truy cập {REAZON_GATE} ({type(e).__name__}). "
              "Hãy đăng nhập và chấp nhận điều khoản (mục 3), rồi chạy lại từ ô này.")
        return False


# Sources actually used; decided here so the data/run hashes (next cell) reflect them.
USE = {"common_voice_ja": DATASETS["common_voice_ja"],
       "reazonspeech": DATASETS["reazonspeech"] and reazon_access()}
print("Nguồn dữ liệu dùng:", [k for k, v in USE.items() if v])

## 4. Mount Google Drive + resume
Mỗi cấu hình dữ liệu/huấn luyện có một mã băm riêng; checkpoint lưu theo mã đó nên đổi cấu hình sẽ không resume nhầm.

Checkpoint được ghi ra ổ local `/content/checkpoints` (`save_total_limit=1`) rồi chép lên **một** thư mục cố định trên Drive
(`ckpt_latest_<mã>`), bản mới thay bản cũ. Khi chạy lại, ô này chép checkpoint từ Drive về `/content` để huấn luyện tiếp.

**Lưu ý dung lượng Drive (bản miễn phí 15 GB):** file bị xoá trên Drive chuyển vào **Thùng rác** và vẫn tính vào hạn mức.
Để giới hạn, checkpoint chỉ được chép lên Drive tối đa một lần mỗi `DRIVE_SYNC_MIN` = 30 phút (lần lưu đầu tiên luôn chép),
tối đa `DRIVE_SYNC_MAX` = 2 lần mỗi phiên, và không chép ở bước cuối (model cuối được lưu ngay sau đó).

Ước tính dung lượng Drive tối đa với cấu hình mặc định: **≈ 6,6 GB** cho phiên đầu tiên, **≈ 9,4 GB** cho một phiên chạy
tiếp (resume, sau khi đã dọn Thùng rác):

| Thành phần | Dung lượng |
|---|---|
| Checkpoint trên Drive (trọng số fp32 242M tham số ≈ 0,97 GB + trạng thái AdamW ≈ 1,94 GB) | ≈ 2,9 GB |
| Bản checkpoint cũ trong Thùng rác (tối đa `DRIVE_SYNC_MAX` − 1 = 1 bản) | ≈ 2,9 GB |
| Model fine-tune cuối (`final_*`, fp16) | ≈ 0,5 GB |
| Bản zip CTranslate2 int8 chép lên Drive | ≈ 0,25 GB |
| Kết quả đánh giá (văn bản) | < 0,01 GB |
| **Tổng phiên đầu** | **≈ 6,6 GB** |
| Phiên chạy tiếp: bản checkpoint có sẵn từ phiên trước cũng bị thay, nên Thùng rác có tới `DRIVE_SYNC_MAX` = 2 bản (thêm ≈ 2,9 GB) | **≈ 9,4 GB** |

Dữ liệu âm thanh **không** lưu lên Drive (chỉ ở `/content`, mất khi phiên kết thúc). Mỗi lần chạy lại (phiên mới) có thể thêm
tối đa 2 × 2,9 GB vào Thùng rác; smoke test (`SMOKE_TEST = True`) có mã băm riêng nên cũng để lại khoảng 3,7 GB. Vì vậy trước
khi chạy lại, hãy vào https://drive.google.com/drive/trash và xoá vĩnh viễn các thư mục `ckpt_latest_*` cũ (hoặc "Dọn sạch
thùng rác"), và xoá các thư mục `ckpt_latest_*`/`final_*` của lần smoke test.

In [ ]:
# CELL: drive
import glob, hashlib, json, re, shutil

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:  # local dry-run
    IN_COLAB = False
    DRIVE_DIR = os.path.abspath(os.environ.get("WHISPER_WORK_DIR", "whisper_work"))
os.makedirs(DRIVE_DIR, exist_ok=True)


def _hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:10]


DATA_CFG = dict(datasets=USE, cv=[CV_REPO, CV_REV, CV_MAX_TRAIN], reazon=[REAZON_REPO, REAZON_REV,
                REAZON_TRAIN_SHARDS, REAZON_MAX_TRAIN], eval=EVAL_PER_SOURCE, sec=[MIN_SEC, MAX_SEC],
                max_label=MAX_LABEL_TOKENS, model=BASE_MODEL, seed=SEED, v=1)
DATA_HASH = _hash(DATA_CFG)
RUN_HASH = _hash(dict(data=DATA_HASH, steps=MAX_STEPS, lr=LR, batch=BATCH, accum=GRAD_ACCUM, fp16=FP16,
                      optim=OPTIM, warmup=WARMUP_STEPS))
LOCAL_DATA = f"/content/whisper_data_{DATA_HASH}" if IN_COLAB else f"{DRIVE_DIR}/data_{DATA_HASH}"  # audio: local only
LOCAL_CKPT = "/content/checkpoints" if IN_COLAB else f"{DRIVE_DIR}/local_ckpt"  # Trainer output_dir (local disk)
DRIVE_CKPT = f"{DRIVE_DIR}/ckpt_latest_{RUN_HASH}"   # the single checkpoint copy kept on Drive
FINAL_DIR = f"{DRIVE_DIR}/final_{RUN_HASH}"
DONE_FLAG = f"{FINAL_DIR}/DONE"                      # written last, after weights + processor files
BASE_EVAL_FILE = f"{DRIVE_DIR}/eval/base_{DATA_HASH}.json"


def _nonempty(f: str) -> bool:
    return os.path.isfile(f) and os.path.getsize(f) > 0


def ckpt_ok(p: str) -> bool:
    # A usable checkpoint has trainer state, non-empty weights and optimizer state.
    return all(_nonempty(f"{p}/{f}") for f in ("trainer_state.json", "model.safetensors", "optimizer.pt"))


def ckpt_step(p: str) -> int:
    with open(f"{p}/trainer_state.json", encoding="utf-8") as f:
        return int(json.load(f)["global_step"])


def last_checkpoint(d: str):
    # Newest complete checkpoint-N in d; incomplete ones (cut off mid-save) are skipped.
    cks = [p for p in glob.glob(f"{d}/checkpoint-*") if re.search(r"checkpoint-(\d+)$", p)]
    for p in sorted(cks, key=lambda p: int(p.rsplit("-", 1)[1]), reverse=True):
        if ckpt_ok(p):
            return p
        print(f"Bỏ qua checkpoint không đầy đủ: {p}")
    return None


def sync_to_drive(ckpt: str):
    # Copy a local checkpoint to DRIVE_CKPT: write a temp folder, then swap by rename (one copy on Drive).
    tmp, old = DRIVE_CKPT + ".tmp", DRIVE_CKPT + ".old"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(ckpt, tmp)
    shutil.rmtree(old, ignore_errors=True)
    if os.path.isdir(DRIVE_CKPT):
        os.rename(DRIVE_CKPT, old)
    os.rename(tmp, DRIVE_CKPT)
    shutil.rmtree(old, ignore_errors=True)
    print(f"Đã chép {os.path.basename(ckpt)} lên {DRIVE_CKPT}")


def restore_from_drive():
    # Copy the Drive checkpoint into LOCAL_CKPT unless a newer local one exists; return the local path.
    local = last_checkpoint(LOCAL_CKPT)
    src = next((p for p in (DRIVE_CKPT, DRIVE_CKPT + ".old") if ckpt_ok(p)), None)  # .old: swap was interrupted
    if src and (local is None or ckpt_step(src) > ckpt_step(local)):
        dst = f"{LOCAL_CKPT}/checkpoint-{ckpt_step(src)}"
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
        print(f"Đã chép checkpoint từ Drive về {dst}")
        local = dst
    return local


TRAIN_DONE = os.path.isfile(DONE_FLAG)
RESUME_CKPT = None if TRAIN_DONE else restore_from_drive()
print(f"Thư mục làm việc: {DRIVE_DIR}")
print(f"Model fine-tune: {'đã xong' if TRAIN_DONE else 'chưa'}; checkpoint để resume: {RESUME_CKPT or 'không có'}")

## 5. Hàm dùng chung: chuẩn hoá CER, lọc câu, lịch chép Drive
CER của model gốc và model fine-tune được tính **giống hệt nhau**: chuẩn hoá NFKC, chữ thường, bỏ toàn bộ dấu câu, ký hiệu và
khoảng trắng ở cả câu dự đoán lẫn câu tham chiếu, rồi lấy tổng số lỗi ký tự / tổng số ký tự tham chiếu trên cả tập test.
`should_sync` là bản sao y hệt hàm trong `training/cleaning.py` (có test trong repo).

In [ ]:
# CELL: helpers

import unicodedata


def norm_cer(text: str) -> str:
    """NFKC + casefold, dropping punctuation, symbols, whitespace and control chars (applied to every model)."""
    t = unicodedata.normalize("NFKC", text).casefold()
    return "".join(c for c in t if unicodedata.category(c)[0] not in "PSZC")


def edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def corpus_cer(hyps, refs) -> float:
    """Corpus CER: sum of char edits / sum of reference chars after norm_cer; empty references are skipped."""
    edits = chars = 0
    for h, r in zip(hyps, refs):
        r = norm_cer(r)
        if r:
            edits += edit_distance(norm_cer(h), r)
            chars += len(r)
    return edits / chars if chars else 0.0


def asr_text_ok(text: str) -> bool:
    """Non-empty after normalisation and not a bracketed on-screen caption (ReazonSpeech has some)."""
    t = text.strip()
    return bool(norm_cer(t)) and not any(c in t for c in "［］[]")


def should_sync(step: int, max_steps: int, last_sync_ts: float | None, now: float, min_minutes: float,
                n_synced: int = 0, max_syncs: int | None = None) -> bool:
    """Whether a checkpoint saved at `step` is copied to Drive.

    Never at/after the last step (the final model is saved right after); the first save always syncs;
    later ones only after min_minutes since the previous sync and while fewer than max_syncs were made.
    """
    if max_steps and step >= max_steps:
        return False
    if max_syncs is not None and n_synced >= max_syncs:
        return False
    return last_sync_ts is None or now - last_sync_ts >= min_minutes * 60

## 6. Tải dữ liệu (đọc dạng luồng, chỉ một tập con nhỏ)
- **Common Voice 17.0 ja**: đọc từ bản parquet `fixie-ai/common_voice_17_0` (Mozilla đã ngừng phát hành Common Voice trên
  Hugging Face từ 10/2025, bản chính thức nay ở Mozilla Data Collective và cần tài khoản). Train = split `train` + `validation`;
  test = split `test` chính thức. Các split của Common Voice không trùng người nói; notebook vẫn kiểm tra lại: câu train của
  người nói (`client_id`) có trong split test bị loại, câu train trùng nội dung câu test cũng bị loại.
- **ReazonSpeech v2 `small`**: notebook kiểm tra bạn đã được cấp quyền ở trang chính thức `reazon-research/reazonspeech`
  (đã chấp nhận điều khoản), rồi đọc dữ liệu từ bản parquet `japanese-asr/whisper_transcriptions.reazonspeech.small` (cùng
  âm thanh 16 kHz và transcript; trang chính thức chỉ có script tải không chạy được với thư viện `datasets` mới). Không có
  id người nói/chương trình, nên tập test lấy từ **file parquet cuối** (không dùng để train) và câu train trùng nội dung câu
  test bị loại.
- File parquet được đọc từng nhóm hàng qua `HfFileSystem` nên chỉ tải phần cần dùng. Âm thanh được giải mã và resample về
  **16 kHz** mono, lọc độ dài 0,5–30 s, lưu dạng int16 ở `/content` (không lưu lên Drive).

In [ ]:
# CELL: data
import io, math, time
import av  # installed with faster-whisper
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from transformers import WhisperProcessor
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="ja", task="transcribe")
tok = processor.tokenizer  # labels: <|startoftranscript|><|ja|><|transcribe|><|notimestamps|> text <|endoftext|>
fs = HfFileSystem()


def parquet_rows(repo, rev, path, columns, limit=None):
    # Stream rows of one Hub parquet file; only the row groups actually read are downloaded.
    n = 0
    with fs.open(f"datasets/{repo}@{rev}/{path}", "rb", block_size=8 << 20) as f:
        for batch in pq.ParquetFile(f).iter_batches(batch_size=64, columns=columns):
            for row in batch.to_pylist():
                if limit is not None and n >= limit:
                    return
                n += 1
                yield row


def shards(repo, rev, pattern):
    root = f"datasets/{repo}@{rev}/"
    return sorted(p[len(root):] if p.startswith(root) else p.split("/", 3)[-1] for p in fs.glob(root + pattern))


def decode(row):
    # Bytes (mp3/flac) -> float32 mono 16 kHz via PyAV; None if undecodable.
    try:
        chunks = []
        rs = av.AudioResampler(format="flt", layout="mono", rate=SAMPLE_RATE)
        with av.open(io.BytesIO(row["audio"]["bytes"])) as c:
            for frame in c.decode(audio=0):
                chunks += [f.to_ndarray().reshape(-1) for f in rs.resample(frame)]
        chunks += [f.to_ndarray().reshape(-1) for f in rs.resample(None)]  # flush
        return np.concatenate(chunks).astype(np.float32) if chunks else None
    except Exception:
        return None


def usable(audio, text):
    # Labels or None: duration window, caption filter, decoder length limit.
    if audio is None or not (MIN_SEC <= len(audio) / SAMPLE_RATE <= MAX_SEC) or not asr_text_ok(text):
        return None
    labels = tok(text.strip()).input_ids
    return labels if len(labels) <= MAX_LABEL_TOKENS else None


class ClipWriter:
    # Appends int16 PCM to <split>.i16 and metadata to <split>.json (memory-mapped later).
    def __init__(self, split):
        self.split, self.index, self.off = split, [], 0
        self.f = open(f"{LOCAL_DATA}/{split}.i16", "wb")

    def add(self, source, text, audio, labels):
        pcm = (np.clip(audio, -1.0, 1.0) * 32767).astype(np.int16)
        self.f.write(pcm.tobytes())
        self.index.append({"off": self.off, "n": len(pcm), "source": source, "text": text.strip(), "labels": labels})
        self.off += len(pcm)

    def close(self):
        self.f.close()
        with open(f"{LOCAL_DATA}/{self.split}.json", "w", encoding="utf-8") as f:
            json.dump(self.index, f, ensure_ascii=False)


def prepare():
    os.makedirs(LOCAL_DATA, exist_ok=True)
    train, test = ClipWriter("train"), ClipWriter("test")
    test_keys, stats = set(), {}

    if USE["common_voice_ja"]:
        t0, src = time.time(), "common_voice_ja"
        test_files = shards(CV_REPO, CV_REV, "ja/test-*.parquet")
        # All speakers of the official test split (column projection: only client_id is downloaded).
        test_spk = {r["client_id"] for p in test_files for r in parquet_rows(CV_REPO, CV_REV, p, ["client_id"])}
        per = math.ceil(EVAL_PER_SOURCE / len(test_files))  # spread test clips over all test shards
        n_te = 0
        for p in test_files:
            if n_te >= EVAL_PER_SOURCE:
                break
            for r in parquet_rows(CV_REPO, CV_REV, p, ["audio", "sentence"], limit=per):
                if n_te >= EVAL_PER_SOURCE:
                    break
                a = decode(r)
                lab = usable(a, r["sentence"])
                if lab:
                    test.add(src, r["sentence"], a, lab)
                    test_keys.add(norm_cer(r["sentence"]))
                    n_te += 1
        n_tr, dropped = 0, {"speaker": 0, "text": 0, "filter": 0}
        for p in shards(CV_REPO, CV_REV, "ja/train-*.parquet") + shards(CV_REPO, CV_REV, "ja/validation-*.parquet"):
            if n_tr >= CV_MAX_TRAIN:
                break
            for r in parquet_rows(CV_REPO, CV_REV, p, ["client_id", "audio", "sentence"]):
                if n_tr >= CV_MAX_TRAIN:
                    break
                if r["client_id"] in test_spk:
                    dropped["speaker"] += 1
                    continue
                if norm_cer(r["sentence"]) in test_keys:
                    dropped["text"] += 1
                    continue
                a = decode(r)
                lab = usable(a, r["sentence"])
                if not lab:
                    dropped["filter"] += 1
                    continue
                train.add(src, r["sentence"], a, lab)
                n_tr += 1
        stats[src] = {"train": n_tr, "test": n_te, "dropped": dropped}
        print(f"[{src}] train {n_tr} / test {n_te} câu, bỏ {dropped} ({time.time() - t0:.0f}s)")

    if USE["reazonspeech"]:
        t0, src = time.time(), "reazonspeech"
        assert REAZON_TRAIN_SHARDS < REAZON_SHARDS
        name = lambda i: f"small/train-{i:05d}-of-{REAZON_SHARDS:05d}.parquet"
        cols = ["name", "audio", "transcription"]
        test_names, n_te = set(), 0
        for r in parquet_rows(REAZON_REPO, REAZON_REV, name(REAZON_SHARDS - 1), cols):
            if n_te >= EVAL_PER_SOURCE:
                break
            a = decode(r)
            lab = usable(a, r["transcription"])
            if lab:
                test.add(src, r["transcription"], a, lab)
                test_keys.add(norm_cer(r["transcription"]))
                test_names.add(r["name"])
                n_te += 1
        n_tr, dropped = 0, {"text": 0, "filter": 0}
        for i in range(REAZON_TRAIN_SHARDS):
            if n_tr >= REAZON_MAX_TRAIN:
                break
            for r in parquet_rows(REAZON_REPO, REAZON_REV, name(i), cols):
                if n_tr >= REAZON_MAX_TRAIN:
                    break
                assert r["name"] not in test_names
                if norm_cer(r["transcription"]) in test_keys:
                    dropped["text"] += 1
                    continue
                a = decode(r)
                lab = usable(a, r["transcription"])
                if not lab:
                    dropped["filter"] += 1
                    continue
                train.add(src, r["transcription"], a, lab)
                n_tr += 1
        stats[src] = {"train": n_tr, "test": n_te, "dropped": dropped}
        print(f"[{src}] train {n_tr} / test {n_te} câu, bỏ {dropped} ({time.time() - t0:.0f}s)")

    train.close()
    test.close()
    assert train.index and test.index, "Không có dữ liệu: bật ít nhất một nguồn và kiểm tra quyền truy cập"
    with open(f"{LOCAL_DATA}/DONE", "w", encoding="utf-8") as f:
        json.dump({"stats": stats, "config": DATA_CFG}, f, ensure_ascii=False)


class ClipDataset:
    # Memory-mapped clips; features are computed on the fly (in DataLoader workers).
    def __init__(self, split):
        self.path = f"{LOCAL_DATA}/{split}.i16"
        with open(f"{LOCAL_DATA}/{split}.json", encoding="utf-8") as f:
            self.index = json.load(f)
        self._mm = None

    def __len__(self):
        return len(self.index)

    def audio(self, i):
        if self._mm is None:
            self._mm = np.memmap(self.path, np.int16, mode="r")
        e = self.index[i]
        return self._mm[e["off"]:e["off"] + e["n"]].astype(np.float32) / 32768

    def __getitem__(self, i):
        feats = processor.feature_extractor(self.audio(i), sampling_rate=SAMPLE_RATE).input_features[0]
        return {"input_features": feats, "labels": self.index[i]["labels"]}


if os.path.isfile(f"{LOCAL_DATA}/DONE"):
    print(f"Đã có dữ liệu ở {LOCAL_DATA}, bỏ qua bước tải.")
else:
    prepare()
TRAIN_DS, TEST_DS = ClipDataset("train"), ClipDataset("test")
hours = sum(e["n"] for e in TRAIN_DS.index) / SAMPLE_RATE / 3600
print(f"Train: {len(TRAIN_DS)} câu ({hours:.1f} giờ); test: {len(TEST_DS)} câu")
for e in TRAIN_DS.index[:3]:
    print(f"  [{e['source']}] {e['text']}")

## 7. Huấn luyện (`Seq2SeqTrainer`)
fp16, batch 8 × tích luỹ gradient 2, gradient checkpointing, lr 1e-5, `MAX_STEPS = 2000` bước. Mỗi `SAVE_STEPS` bước,
checkpoint được lưu ra `/content` rồi (theo lịch giới hạn ở mục 4) chép đè lên một thư mục duy nhất trên Drive; chạy lại
notebook sau khi mất kết nối sẽ tiếp tục từ checkpoint đó. Model cuối được coi là xong chỉ khi có file `DONE` (ghi sau cùng).

In [ ]:
# CELL: train
import torch
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          WhisperForConditionalGeneration, set_seed)

START_ID = tok.convert_tokens_to_ids("<|startoftranscript|>")


def collate(batch):
    feats = torch.tensor(np.stack([b["input_features"] for b in batch]), dtype=torch.float32)
    m = max(len(b["labels"]) for b in batch)
    lab = torch.full((len(batch), m), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        lab[i, :len(b["labels"])] = torch.tensor(b["labels"], dtype=torch.long)
    if bool((lab[:, 0] == START_ID).all()):
        lab = lab[:, 1:]  # the model prepends decoder_start_token_id when shifting labels right
    return {"input_features": feats, "labels": lab}


class DriveSync(TrainerCallback):
    # After a local save, replace the single Drive copy with it (rate-limited, capped, never at the final step).

    def __init__(self):
        self.last_ts, self.n = None, 0

    def on_save(self, args, state, control, **kwargs):
        ckpt = f"{args.output_dir}/checkpoint-{state.global_step}"
        if not (state.is_world_process_zero and ckpt_ok(ckpt)):
            return
        now = time.time()
        if should_sync(state.global_step, state.max_steps, self.last_ts, now, DRIVE_SYNC_MIN, self.n, DRIVE_SYNC_MAX):
            sync_to_drive(ckpt)
            self.last_ts, self.n = now, self.n + 1
        else:
            print(f"Checkpoint {state.global_step} chỉ lưu local (chưa đến lượt chép lên Drive)")


def free_gpu():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if TRAIN_DONE:
    print(f"Model fine-tune đã có ở {FINAL_DIR}, bỏ qua huấn luyện.")
else:
    set_seed(SEED)
    model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float32)  # fp32 master weights
    model.config.use_cache = False  # required with gradient checkpointing
    args = Seq2SeqTrainingArguments(
        output_dir=LOCAL_CKPT, max_steps=MAX_STEPS, learning_rate=LR,
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        fp16=FP16 and torch.cuda.is_available(), gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}, optim=OPTIM, warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="linear", logging_steps=25, save_strategy="steps", save_steps=SAVE_STEPS,
        save_total_limit=1, report_to="none", remove_unused_columns=False, dataloader_num_workers=2, seed=SEED)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=TRAIN_DS, data_collator=collate,
                             callbacks=[DriveSync()])
    ckpt = restore_from_drive()
    print(f"Bắt đầu huấn luyện {'tiếp từ ' + ckpt if ckpt else 'từ đầu'}")
    trainer.train(resume_from_checkpoint=ckpt)
    # Processor files first (unchanged by fine-tuning), weights next, DONE sentinel last,
    # so an interrupted save is never taken for a finished model.
    os.makedirs(FINAL_DIR, exist_ok=True)
    processor.save_pretrained(FINAL_DIR)  # tokenizer.json, preprocessor_config.json, ...
    trainer.model.config.use_cache = True
    trainer.model.half().save_pretrained(FINAL_DIR)  # fp16 (~0.5 GB) + generation_config.json
    assert all(os.path.isfile(f"{FINAL_DIR}/{f}") for f in ("tokenizer.json", "preprocessor_config.json"))
    with open(DONE_FLAG, "w") as f:
        f.write(f"step {trainer.state.global_step}")
    TRAIN_DONE = True
    del trainer, model
    free_gpu()
    print(f"Đã lưu model fine-tune vào {FINAL_DIR}")

## 8. Đánh giá: CER model gốc vs fine-tune
Cả hai model chạy trên **cùng** tập test giữ riêng, cùng cấu hình sinh (`language="ja"`, `task="transcribe"`, greedy
`num_beams=1` giống app, fp16) và cùng hàm chuẩn hoá CER (mục 5). CER càng thấp càng tốt. Kết quả lưu trên Drive nên chạy lại
không phải nhận dạng lại.

In [ ]:
# CELL: eval
GEN_KW = dict(language="ja", task="transcribe", num_beams=1, max_new_tokens=225)  # identical for both models
DEV = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEV == "cuda" else torch.float32
REFS = [e["text"] for e in TEST_DS.index]
SOURCES = sorted({e["source"] for e in TEST_DS.index})


def hf_transcribe(path):
    model = WhisperForConditionalGeneration.from_pretrained(path, dtype=DTYPE).to(DEV).eval()
    out = []
    for b in range(0, len(TEST_DS), EVAL_BATCH):
        audios = [TEST_DS.audio(i) for i in range(b, min(b + EVAL_BATCH, len(TEST_DS)))]
        feats = processor.feature_extractor(audios, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_features
        with torch.no_grad():
            ids = model.generate(input_features=feats.to(DEV, DTYPE), **GEN_KW)
        out += [t.strip() for t in tok.batch_decode(ids, skip_special_tokens=True)]
    del model
    free_gpu()
    return out


def cached(file, key, fn):
    data = {}
    if os.path.isfile(file):
        with open(file, encoding="utf-8") as f:
            data = json.load(f)
    if key not in data or len(data[key]) != len(REFS):
        data[key] = fn()
        os.makedirs(os.path.dirname(file), exist_ok=True)
        with open(file, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
    return data[key]


def cer_table(hyps: dict):
    names = list(hyps)
    print(f"{'Bộ test':<20}{'n':>6}" + "".join(f"{'CER ' + n:>18}" for n in names))
    for src in SOURCES + ["tất cả"]:
        idx = [i for i, e in enumerate(TEST_DS.index) if src == "tất cả" or e["source"] == src]
        row = "".join(f"{100 * corpus_cer([hyps[n][i] for i in idx], [REFS[i] for i in idx]):>17.2f}%" for n in names)
        print(f"{src:<20}{len(idx):>6}{row}")


HYP = {"gốc": cached(BASE_EVAL_FILE, "hf", lambda: hf_transcribe(BASE_MODEL)),
       "fine-tune": cached(f"{FINAL_DIR}/eval_test.json", "hf", lambda: hf_transcribe(FINAL_DIR))}
cer_table(HYP)
for i in range(min(8, len(REFS))):
    print(f"[{i + 1}] Tham chiếu: {REFS[i]}\n    Gốc      : {HYP['gốc'][i]}\n    Fine-tune: {HYP['fine-tune'][i]}")

## 9. Xuất faster-whisper (CTranslate2 int8) + kiểm tra như app
`ct2-transformers-converter --quantization int8` tạo thư mục `small-ja/` gồm `model.bin`, `config.json`,
`vocabulary.json`, `tokenizer.json`, `preprocessor_config.json` — đúng cấu trúc faster-whisper cần. Tên thư mục chỉ gồm chữ,
số, `_`, `.`, `-` (≤ 64 ký tự) để app chấp nhận làm `asr.model`.

Ô này cũng tính CER bằng faster-whisper int8 theo đúng cách app gọi (`beam_size=1`, `vad_filter=False`,
`condition_on_previous_text=False`) cho model `small` gốc của app (Systran/faster-whisper-small) và model vừa xuất.

In [ ]:
# CELL: export
import subprocess
from faster_whisper import WhisperModel

OUT_ROOT = "/content/export" if IN_COLAB else f"{DRIVE_DIR}/export"
OUT_DIR = f"{OUT_ROOT}/{EXPORT_NAME}"
assert re.fullmatch(r"[\w.\-]{1,64}", EXPORT_NAME)  # same rule as asr_model in backend/server/protocol.py
shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_ROOT, exist_ok=True)
subprocess.run(["ct2-transformers-converter", "--model", FINAL_DIR, "--output_dir", OUT_DIR,
                "--quantization", "int8",
                "--copy_files", "tokenizer.json", "preprocessor_config.json"], check=True)
print("Các file:", sorted(os.listdir(OUT_DIR)))
for f in ("model.bin", "config.json", "vocabulary.json", "tokenizer.json", "preprocessor_config.json"):
    assert os.path.isfile(f"{OUT_DIR}/{f}"), f


def fw_transcribe(path):
    # Same call as backend/asr/whisper.py (language fixed to ja).
    try:
        m = WhisperModel(path, device="cuda", compute_type="int8")
    except Exception as e:
        print(f"CUDA lỗi ({e}), dùng CPU")
        m = WhisperModel(path, device="cpu", compute_type="int8")
    out = []
    for i in range(len(TEST_DS)):
        segs, _ = m.transcribe(TEST_DS.audio(i), language="ja", task="transcribe", vad_filter=False,
                               beam_size=1, condition_on_previous_text=False)
        out.append("".join(s.text.strip() for s in segs).strip())
    del m
    free_gpu()
    return out


CT2_HYP = {"gốc int8": cached(BASE_EVAL_FILE, "ct2", lambda: fw_transcribe(BASE_SIZE)),
           "fine-tune int8": fw_transcribe(OUT_DIR)}
cer_table(CT2_HYP)

## 10. Nén zip và tải về
Các file nằm ngay ở gốc zip. Trên máy Windows: tạo thư mục `models/asr/small-ja/` rồi giải nén `whisper-ja-ct2-int8.zip`
**vào `models/asr/small-ja/`** sao cho có `models/asr/small-ja/model.bin`, rồi đặt `"asr": {"model": "small-ja"}` trong
`config.local.json` (hoặc chọn `small-ja` trong ứng dụng nếu có). Muốn quay lại model gốc thì đổi về `"small"`. Bản zip cũng được chép lên Drive phòng khi tải trực tiếp bị lỗi.

In [ ]:
# CELL: zip
import zipfile

# Files at the zip root (like the NLLB export); written next to OUT_DIR, not inside it.
zip_path = shutil.make_archive(f"{OUT_ROOT}/{ZIP_NAME[:-4]}", "zip", root_dir=OUT_DIR)
with zipfile.ZipFile(zip_path) as z:
    print(sorted(z.namelist()))
print(f"{zip_path}: {os.path.getsize(zip_path) / 2**20:.0f} MB")
if IN_COLAB:
    shutil.copy(zip_path, DRIVE_DIR)
    from google.colab import files
    files.download(zip_path)